# Aula 3 · Lab 4 — Replicação e falhas de broker

## Objetivo
Até aqui o cluster esteve saudável. Agora vamos **derrubar brokers de verdade** com um produtor escrevendo o
tempo todo, para observar:

- a **eleição de um novo líder** quando o líder de uma partição cai;
- o **ISR** encolhendo e voltando a crescer;
- a combinação `acks=all` + `min.insync.replicas=2` aceitando escritas com 1 broker fora e **recusando** com 2 fora;
- a **recuperação** automática quando os brokers voltam.

## Como os brokers são derrubados
O Jupyter roda dentro de um container e não controla os outros. Os comandos que param e religam brokers rodam
no **Azure Cloud Shell**, de dentro da pasta do material:
```bash
bash infra/vm.sh "docker stop kafka2"     # derruba o broker 2
bash infra/vm.sh "docker start kafka2"    # religa
```
O `infra/vm.sh` executa o comando dentro da VM pela própria Azure (`az vm run-command`), sem SSH.
Cada chamada leva de 20 a 40 segundos para voltar.

`docker stop` encerra o processo do broker: para o resto do cluster, é como se a máquina tivesse desligado.

## Etapa 0 — Tópico de teste e situação inicial

Criamos `aula3-replicado` com 3 partições, replicação 3 e **`min.insync.replicas=2`**.

**Por que essa combinação?** Com `acks=all`, uma escrita só é aceita se pelo menos `min.insync.replicas`
réplicas do ISR a gravarem. Com 3 réplicas e mínimo 2:

| Brokers vivos | Réplicas no ISR | Escritas com `acks=all` | Por quê |
|---|---|---|---|
| 3 | 3 | aceitas | 3 ≥ 2 |
| 2 | 2 | aceitas | 2 ≥ 2: o cluster tolera 1 falha sem parar |
| 1 | 1 | **recusadas** (`NOT_ENOUGH_REPLICAS`) | 1 < 2: o Kafka prefere ficar indisponível para escrita a aceitar dados guardados numa única cópia |

A função `descrever()` mostra os brokers vivos e, para cada partição, o líder, as réplicas e o ISR (campos
explicados no Lab 1). **Situação esperada agora:** 3 brokers vivos e ISR com 3 réplicas em todas as partições.

In [ ]:
import json, time, threading
from confluent_kafka import Producer
from confluent_kafka.admin import AdminClient, NewTopic
from comum.config import BOOTSTRAP

admin = AdminClient({"bootstrap.servers": BOOTSTRAP})
TOPICO = "aula3-replicado"
try:
    admin.create_topics([NewTopic(TOPICO, 3, 3, config={"min.insync.replicas": "2"})])[TOPICO].result()
except Exception as e:
    print("aviso:", e)

def descrever():
    meta = admin.list_topics(topic=TOPICO, timeout=10)
    vivos = sorted(meta.brokers)
    print(time.strftime("%H:%M:%S"), "brokers vivos:", vivos)
    for p in sorted(meta.topics[TOPICO].partitions.values(), key=lambda p: p.id):
        print(f"  partição {p.id}: líder={p.leader} réplicas={p.replicas} ISR={p.isrs}")

descrever()

**Anote quem é o líder de cada partição.** Na próxima etapa, as partições lideradas pelo broker 2 vão trocar de líder.

## Etapa 1 — Um produtor contínuo em segundo plano

Esta célula inicia uma *thread* que envia **5 mensagens por segundo** durante todo o lab e conta sucessos e erros
no dicionário `estatisticas`. Os valores são **acumulados** desde o início, então compare a variação entre uma leitura e outra.

| Parâmetro | Valor | Por que importa aqui |
|---|---|---|
| `acks` | `all` | Cada mensagem precisa ser gravada por todas as réplicas do ISR, respeitando o `min.insync.replicas` |
| `enable.idempotence` | `True` | Os reenvios automáticos durante a troca de líder não geram duplicatas |
| `request.timeout.ms` | 5 000 | Quanto tempo esperar a resposta de uma requisição ao broker antes de considerá-la perdida e tentar de novo |
| `delivery.timeout.ms` | 10 000 | Prazo **total** para entregar a mensagem, somando todas as tentativas. Passado esse tempo, o callback recebe erro. Usamos 10 s, em vez dos 120 s padrão, para os erros aparecerem rápido durante a aula |

Entre o broker cair e o novo líder ser eleito, o produtor **reenvia** as mensagens automaticamente
(*retries*). Se a troca acontecer dentro do `delivery.timeout.ms`, a aplicação nem percebe a falha.

In [ ]:
estatisticas = {"ok": 0, "erro": 0, "ultimo_erro": None}
rodando = threading.Event()
rodando.set()

def produzir():
    p = Producer({"bootstrap.servers": BOOTSTRAP, "acks": "all", "enable.idempotence": True,
                  "delivery.timeout.ms": 10000, "request.timeout.ms": 5000})
    def cb(err, msg):
        if err:
            estatisticas["erro"] += 1
            estatisticas["ultimo_erro"] = str(err)
        else:
            estatisticas["ok"] += 1
    i = 0
    while rodando.is_set():
        p.produce(TOPICO, key=f"k{i % 9}", value=json.dumps({"i": i}), callback=cb)
        p.poll(0.2)
        i += 1
    p.flush(15)

threading.Thread(target=produzir, daemon=True).start()
time.sleep(3)
print(estatisticas)

**Resultado esperado:** `ok` com cerca de 15 (3 s × 5 msg/s) e `erro: 0`.

## Etapa 2 — Derrube um broker

No **Cloud Shell**:
```bash
bash infra/vm.sh "docker stop kafka2"
```
Quando o comando voltar, espere ~10 segundos e rode a célula abaixo. Repita algumas vezes.

**O que acontece por baixo:**
1. o controller percebe que o broker 2 parou de enviar sinais de vida (*heartbeats*);
2. para cada partição em que o broker 2 era **líder**, o controller promove outra réplica do **ISR**. Como ela já
   estava em dia, nenhuma mensagem confirmada se perde;
3. o broker 2 é retirado do ISR de todas as partições;
4. os clientes recebem metadados novos e passam a falar com os novos líderes.

**O que observar na saída:**
- `brokers vivos` passa a mostrar só `[1, 3]`;
- nas partições que eram lideradas pelo 2, o `líder` agora é 1 ou 3;
- o 2 continua na lista de `réplicas` (ele ainda **é** dono de uma cópia), mas **sumiu do `ISR`**;
- `ok` continua subindo e `erro` continua em 0: com ISR = 2 ≥ `min.insync.replicas` = 2, as escritas são aceitas.

In [ ]:
descrever()
print(estatisticas)

## Etapa 3 — Derrube um segundo broker

No Cloud Shell:
```bash
bash infra/vm.sh "docker stop kafka3"
```
Agora só o broker 1 está vivo. Espere ~15 segundos e rode a célula abaixo.

**O que esperar:**
- o ISR de cada partição cai para **1 réplica**, abaixo do `min.insync.replicas=2`;
- o líder **recusa** as escritas com `acks=all`. O produtor tenta até vencer o `delivery.timeout.ms` (10 s), e então
  o callback recebe erro: `erro` passa a subir e `ultimo_erro` mostra algo como `NOT_ENOUGH_REPLICAS`, `REQUEST_TIMED_OUT`
  ou `Message timed out`;
- `ok` **para de subir**.

**Por que o Kafka recusa em vez de aceitar?** Se aceitasse com uma única cópia e esse broker também falhasse,
os dados confirmados ao produtor seriam perdidos. O `min.insync.replicas` troca **disponibilidade** por
**durabilidade**: é uma escolha do lado *CP* do teorema CAP.

> **O quórum KRaft também se perde.** Os 3 brokers são os controladores do cluster, e o quórum precisa da maioria
> (2 de 3) para tomar decisões. Com só 1 vivo, nenhum novo líder pode ser eleito e os metadados podem ficar
> desatualizados ou indisponíveis. Por isso a célula pode mostrar informações antigas ou falhar ao ler metadados,
> e isso faz parte da lição.

In [ ]:
try:
    descrever()
except Exception as e:
    print("não consegui ler metadados:", e)
print(estatisticas)

## Etapa 4 — Recuperação

No Cloud Shell:
```bash
bash infra/vm.sh "docker start kafka2 kafka3"
```
Espere ~20 segundos e rode a célula abaixo. Repita depois de mais alguns segundos se o ISR ainda não estiver completo.

**O que acontece por baixo:**
1. os brokers voltam e o quórum KRaft se refaz;
2. cada réplica que voltou **copia do líder** as mensagens que perdeu enquanto estava fora (*catch-up*);
3. ao alcançar o líder, ela **volta para o ISR**;
4. o produtor volta a ter sucesso: `ok` sobe de novo e `erro` para.

**Liderança:** logo após a volta, o broker 1 pode continuar líder de quase tudo, e a carga fica desequilibrada.
O Kafka verifica periodicamente se cada partição está com o seu **líder preferido**, o primeiro da lista de réplicas,
e devolve a liderança a ele (*preferred leader election*). Essa verificação roda a cada 5 minutos por padrão
(`leader.imbalance.check.interval.seconds`), então o equilíbrio pode demorar para aparecer.

In [ ]:
descrever()
print(estatisticas)

## Etapa 5 — Encerrando o produtor
Para a thread do produtor e mostra os totais finais. A soma `ok + erro` é o total de mensagens que o produtor tentou enviar.
Os `erro` correspondem ao período da Etapa 3, em que o cluster recusou escritas para proteger os dados.

In [ ]:
rodando.clear()   # para o produtor de fundo
time.sleep(2)
print("final:", estatisticas)

## Para discutir
1. Com `acks=1`, o que poderia ser **perdido** se o líder caísse logo depois de confirmar a escrita, antes de os seguidores copiarem?
2. Por que `replication_factor=3` + `min.insync.replicas=2` é a configuração mais comum em produção? O que mudaria com `min.insync.replicas=1` ou `=3`?
3. Qual é a diferença entre um broker **fora do ar** e um broker **fora do ISR**? Um broker pode estar vivo e fora do ISR?
4. Aqui os 3 brokers estão na **mesma VM**. O que isso significa para a tolerância a falhas de verdade?
   Ligue com *rack awareness* (`broker.rack`) e zonas de disponibilidade da nuvem.